<a href="https://colab.research.google.com/github/iprachuk/Significance_tools/blob/main/Significance_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **1. Imports and Environment Setup**

This section handles the configuration of the working environment. We import essential libraries for data manipulation (`pandas`, `numpy`) and statistical testing (`statsmodels`). Additionally, we mount Google Drive to access the project datasets.

In [ ]:
#imports
from google.colab import drive
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

#connection to google drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Mate/Collaborative/Files


Mounted at /content/drive
/content/drive/MyDrive/Mate/Collaborative/Files


## **2. Data Preparation and Metric Configuration**

In this stage, we load the primary dataset and define the testing framework:
* **Dataset Loading**: Importing the raw data from Google Drive.
* **Metric Mapping**: Configuring specific events (numerators and denominators) to calculate various conversion rates (e.g., *add_to_cart_per_session*).
* **Helper Functions**: Implementing `get_event_value` to efficiently filter and aggregate event sums for specific tests and groups.

In [ ]:
# Завантаження основного датасету з Google Drive
df = pd.read_csv("Portfolio_Project_2.csv")
event_names = np.array(df['event_name'].unique())

# Конфігурація метрик: визначаємо пари подій (чисельник/знаменник) для розрахунку конверсії
metrics_config = [
    {
        'metric_name': 'add_payment_info_per_session',
        'numerator_event': 'add_payment_info',
        'denominator_event': 'session'
    },
    {
        'metric_name': 'add_shipping_info_per_session',
        'numerator_event': 'add_shipping_info',
        'denominator_event': 'session'
    },
    {
        'metric_name': 'begin_checkout_per_session',
        'numerator_event': 'begin_checkout',
        'denominator_event': 'session'
    },
    {
        'metric_name': 'new_accounts_per_session',
        'numerator_event': 'new account',
        'denominator_event': 'session'
    },
    {
        'metric_name': 'conversion_orders_per_session',
        'numerator_event': 'session with orders',
        'denominator_event': 'session'
    },
    {
        'metric_name': 'add_to_cart_per_session',
        'numerator_event': 'add_to_cart',
        'denominator_event': 'session'
    }
]

# Допоміжна функція: фільтрація даних та повернення суму значень для конкретної групи та події(group by реалізований через функцію)
def get_event_value(data, test, test_group, event_name):
  filtered_data = data[
      (data['test'] == test) &
      (data['test_group'] == test_group) &
      (data['event_name'] == event_name)
  ]
  if filtered_data.empty:
    return 0
  return filtered_data['value'].sum()

## **Statistical Analysis (Z-test)**

In this section, we define the logic for the statistical test. We use the **Z-test for proportions** to compare the Control and Test groups. The function also checks for sample validity based on minimum size and success counts to ensure reliable results.

In [ ]:
# Основна функція для проведення A/B тесту (Z-тест пропорцій)
def calculate_metric_result(data, test, metric_config_item):
  numerator_name = metric_config_item["numerator_event"]
  denominator_name = metric_config_item["denominator_event"]

  # Отримуємо дані для контрольної групи (ID 1)
  numerator_control = get_event_value(data, test, 1, numerator_name)
  denominator_control = get_event_value(data, test, 1, denominator_name)

  # Отримуємо дані для тестової групи (ID 2)
  numerator_test = get_event_value(data, test, 2, numerator_name)
  denominator_test = get_event_value(data, test, 2, denominator_name)

  # Розрахунок конверсій (CR) для обох груп
  conversion_rate_control = numerator_control / denominator_control if denominator_control != 0 else 0
  conversion_rate_test = numerator_test / denominator_test if denominator_test != 0 else 0

  # Відсоткова зміна між групами (ліфт)
  metric_change = ((conversion_rate_test - conversion_rate_control) / conversion_rate_control * 100) if conversion_rate_control != 0 else 0

  # Параметри стат. значущості та обмеження для валідності вибірки
  alpha = 0.05
  MIN_SAMPLE_SIZE = 100
  MIN_SUCCESS_COUNT = 5

  # Перевірка, чи достатньо даних для проведення тесту
  if (
    denominator_control == 0 or
    denominator_test == 0 or
    numerator_control > denominator_control or
    numerator_test > denominator_test or
    denominator_control < MIN_SAMPLE_SIZE or
    denominator_test < MIN_SAMPLE_SIZE or
    numerator_control < MIN_SUCCESS_COUNT or
    numerator_test < MIN_SUCCESS_COUNT
):
    z_statistic, p_value, is_significant = None, None, False
    sample_valid = False
  else:
    # Виконання Z-тесту для двох незалежних часток (пропорцій)
    z_statistic, p_value = proportions_ztest(
        [numerator_test, numerator_control],
        [denominator_test, denominator_control]
    )
    is_significant = p_value < alpha
    sample_valid = True

  return {
        'test_number': test,
        'metric': metric_config_item['metric_name'],
        'numerator_event': numerator_name,
        'denominator_event': denominator_name,
        'numerator_control': numerator_control,
        'denominator_control': denominator_control,
        'conversion_rate_control': conversion_rate_control,
        'numerator_test': numerator_test,
        'denominator_test': denominator_test,
        'conversion_rate_test': conversion_rate_test,
        'metric_change': metric_change,
        'z_stat': z_statistic,
        'p_value': p_value,
        'significant': is_significant,
        'sample_valid': sample_valid
    }

## **Segmentation and Execution**

This block automates the calculation across all tests and various segments (device, country, continent, channel). Finally, it filters for valid samples and exports the processed data to a CSV file for visualization in Tableau.

In [ ]:
#array for add result
all_results = []

# Отримуємо перелік всіх унікальних експериментів
unique_tests = df['test'].unique()
# Список сегментів для аналізу
segments = ['device', 'country', 'continent', 'channel']

# Основний цикл обробки по кожному тесту
for current_test in unique_tests:
    test_data = df[df['test'] == current_test]

    # 1. Загальні результати (TOTAL)
    for current_metric in metrics_config:
        metric_result = calculate_metric_result(test_data, current_test, current_metric)
        metric_result['segment_type'] = 'total'
        metric_result['segment_value'] = 'total'
        all_results.append(metric_result)

    # 2. Аналіз у розрізі сегментів
    for current_segment in segments:
        segment_values = test_data[current_segment].dropna().unique()

        for value in segment_values:
            segment_filtered_data = test_data[test_data[current_segment] == value]

            for current_metric in metrics_config:
                metric_result = calculate_metric_result(
                    segment_filtered_data,
                    current_test,
                    current_metric
                )
                metric_result['segment_type'] = current_segment
                metric_result['segment_value'] = value
                all_results.append(metric_result)

# Формування підсумкового DataFrame та збереження
result_df = pd.DataFrame(all_results)
result_df = result_df[result_df['sample_valid'] == True]

# Експорт для Tableau
result_df.to_csv(
    '/content/drive/MyDrive/Mate/Collaborative/Files/AB_testing_tool.csv',
    index=False
)

display(result_df.head())

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_test,denominator_test,conversion_rate_test,metric_change,z_stat,p_value,significant,sample_valid,segment_type,segment_value
0,4,add_payment_info_per_session,add_payment_info,session,3731,105079,0.035507,3601,105141,0.034249,-3.541234,-1.571106,0.116158,False,True,total,total
1,4,add_shipping_info_per_session,add_shipping_info,session,5128,105079,0.048801,4956,105141,0.047137,-3.411125,-1.785795,0.074132,False,True,total,total
2,4,begin_checkout_per_session,begin_checkout,session,12555,105079,0.119482,12267,105141,0.116672,-2.351523,-1.995998,0.045934,True,True,total,total
3,4,new_accounts_per_session,new account,session,8984,105079,0.085498,8687,105141,0.082622,-3.362896,-2.375457,0.017527,True,True,total,total
4,4,conversion_orders_per_session,session with orders,session,10596,105079,0.100838,10481,105141,0.099685,-1.143644,-0.880234,0.378732,False,True,total,total


## **Conclusions and Development Summary**

### **General Summary**
Throughout this project, an automated A/B testing tool was implemented, which includes:
1.  **Data Processing Pipeline**: Automatic loading, aggregation, and conversion calculation for various metrics.
2.  **Statistical Validation**: Implementation of the Z-test for proportions with statistical significance (p-value) checks.
3.  **Deep Segmentation**: The ability to analyze results not only globally but also across devices, countries, and traffic channels.

### **Results**
*   Developed stable code that minimizes manual analytical work.
*   Data is prepared and exported in [**CSV-file**](https://drive.google.com/file/d/1bKe0avpUeVebFpXAmQYeNpdlqoNGIeVA/view?usp=sharing)
 format, ready to be integrated into Tableau dashboards ([**Significance tools**](https://public.tableau.com/app/profile/ihor.prachuk/viz/AB_significance_tools/Significance?publish=yes)).
*   The algorithm accounts for sample size limitations, helping to avoid false-positive results on low data volumes.
*   Created [**A/B test instrument**](https://public.tableau.com/app/profile/ihor.prachuk/viz/ABtest_17752057474510/ABtestinstrument) with which we can perfom a full analysis of the concuted test.



